In [1]:
import os
import time
import random
import warnings
from datetime import date

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree # type: ignore <- pylance milně hlásí chybu
from pathlib import Path
import time
import sys
import polars as pl
import polars.selectors as cs
import json
import pickle
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patheffects as path_effects
from ydata_profiling import ProfileReport
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from pyod.models.iforest import IForest
from pyod.models.ecod import ECOD


current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
sys.path.append(str(current_dir.parent))
from utils import *
from schemas import *
from clean import *
from visualisation_utils import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')
# os.chdir(r'C:\Users\adamp\Projects\CVUT_BAP')
SEED=42
PRINT = True

# Nacteni dat
- pro anomalie neni VIN brano v potaz

In [2]:
lf_prohlidky = pl.scan_parquet(r"E:\CVUT_BAP\kod\data\processed\mereni.parquet")
lf_mereni = pl.scan_parquet(r"E:\CVUT_BAP\kod\data\processed\prohlidky.parquet")
lf = lf_prohlidky.join(lf_mereni, on='CisloProtokolu', how='inner')

In [3]:
# print(lf.collect_schema().names())

In [4]:
df = lf.filter(pl.col('Vysledek_Vyhovuje') == True).drop(['Vozidlo_Vin', 'Vysledek_Vyhovuje']).collect().sample(n=100_000, seed=SEED)

In [5]:
df.write_parquet(r"E:\CVUT_BAP\kod\data\processed\anomaly_sample.parquet")

# Priprava dat pro analyzu

In [6]:
def preprocess_anomalies_pipeline(df, freq_threshold=0.001, custom_cat_cols=None):
    # Zachování ID protokolů pro pozdější identifikaci nálezů
    protocol_ids = df.select("CisloProtokolu")
    df_working = df.drop("CisloProtokolu")

    # Převod seznamů na numerický počet prvků (např. počet závad)
    list_cols = [n for n, t in zip(df_working.columns, df_working.dtypes) if t == pl.List]
    df_working = df_working.with_columns([
        pl.col(c).list.len().fill_null(0).alias(f"{c}_count") for c in list_cols
    ]).drop(list_cols)

    print('1')

    # Identifikace sloupců pro kategorické kódování
    base_cats = [n for n, t in zip(df_working.columns, df_working.dtypes) if t in [pl.String, pl.Categorical]]
    cat_cols = list(set(base_cats + custom_cat_cols)) if custom_cat_cols else base_cats
    
    total_rows = df_working.height
    for col in cat_cols:
        if col not in df_working.columns:
            continue
        print(col)
            
        # Výpočet relativních frekvencí výskytu hodnot
        counts = df_working.group_by(col).len().with_columns(
            (pl.col("len") / total_rows).alias("rel_freq")
        )
        
        # Filtrace dominantních kategorií pro binární příznaky
        top_categories = counts.filter(pl.col("rel_freq") >= freq_threshold).select(col).to_series().to_list()
        
        if len(top_categories) > 0:
            for cat in top_categories:
                # Ošetření null hodnot a formátování názvů nových sloupců
                if cat is None:
                    condition = pl.col(col).is_null()
                    safe_name = "null"
                else:
                    condition = pl.col(col) == cat
                    safe_name = str(cat).replace(" ", "_").replace("-", "_").lower()
                
                df_working = df_working.with_columns(
                    condition.cast(pl.Int8).alias(f"{col}_is_{safe_name}")
                )
        
        # Přidání rarity score zachycující vzácnost výskytu
        df_working = df_working.join(counts.select([col, "rel_freq"]), on=col, how="left")
        df_working = df_working.with_columns(
            pl.col("rel_freq").fill_null(0.0).alias(f"{col}_rarity_score")
        ).drop([col, "rel_freq"])

    print('2')
    # Normalizace logických hodnot na numerické formáty
    df_working = df_working.with_columns([
        pl.col(pl.Boolean).fill_null(False).cast(pl.Int8)
    ])

    # Výběr všech číselných sloupců pro finální škálování
    numeric_cols = [n for n, t in zip(df_working.columns, df_working.dtypes) if t.is_numeric()]
    
    # Odstranění záporných hodnot z reálných měření
    df_working = df_working.with_columns([
        pl.col(c).clip(lower_bound=0) for c in numeric_cols
    ])
    print('3')

    # Konverze na Numpy matici
    X = df_working.select(numeric_cols).to_numpy()
    
    # První stupeň škálování na sjednocený rozsah
    mms = MinMaxScaler(feature_range=(0, 1))
    X_transformed = mms.fit_transform(X)
    print('4')
    
    # Geometrické oddělení chybějících údajů konstantou pod minimem
    X_transformed[np.isnan(X_transformed)] = -1
    
    # Finální standardizace distribucí pro výpočet hlavních komponent
    ss = StandardScaler()
    X_final = ss.fit_transform(X_transformed)
    
    return X_final, protocol_ids

# Seznam číselných sloupců s kategorickým významem
cat_numeric_cols = [
    'Prohlidka_OdpovednaOsoba', 
    'Prohlidka_Stanice_Cislo', 
    'Emise_OdpovednaOsoba', 
    'Vysledek_RidiciJednotkaStav', 
    'Vysledek_Mil', 
    'Obd_KontrolaMil'
] + [col for col in df.columns if col.endswith('_Vysledek') and df[col].dtype.is_numeric()]

# Transformace datové sady
X, id = preprocess_anomalies_pipeline(df, custom_cat_cols=cat_numeric_cols)

1
Nafta_Mereni2_TPS_Vysledek
Nafta_Mereni3_Kourivost_Vysledek
Prohlidka_Stanice_Obec
Benzin_OtackyZvysene_O2_Vysledek
Nafta_MereniPrumer_Kourivost_Vysledek
Prohlidka_OdpovednaOsoba
Nafta_Mereni2_Kourivost_Vysledek
Nafta_MereniPrumer_TPS_Vysledek
Nafta_MereniPrumer_Teplota_Vysledek
Obd_KomunikacniProtokol
Nafta_Mereni2_CasAkcelerace_Vysledek
Benzin_OtackyVolnobezne_N_Vysledek
Nafta_Mereni1_Kourivost_Vysledek
Emise_OdpovednaOsoba
Nafta_Mereni3_CasAkcelerace_Vysledek
MericiPristroj_Verze
Benzin_OtackyVolnobezne_TPS_Vysledek
Nafta_Mereni0_Teplota_Vysledek
Benzin_OtackyZvysene_CO2_Vysledek
Benzin_OtackyVolnobezne_CO2_Vysledek
Nafta_Mereni3_Teplota_Vysledek
Nafta_Mereni0_TPS_Vysledek
Nafta_Mereni2_OtackyPrebehove_Vysledek
Nafta_Mereni1_OtackyVolnobezne_Vysledek
Benzin_OtackyVolnobezne_NOX_Vysledek
Nafta_Mereni3_TPS_Vysledek
Nafta_Mereni2_Teplota_Vysledek
Nafta_Mereni1_OtackyPrebehove_Vysledek
Emise_ZakladniPalivo
Vozidlo_Znacka
Registrace_Stat
Benzin_OtackyVolnobezne_O2_Vysledek
Nafta_Mereni

In [7]:
X.shape

(100000, 2236)

In [8]:
# 1. Redukce dimenzionality (PCA)
print("Zahajuji PCA pro redukci dimenzionality...")
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)
print(f"PCA dokončeno. Vysvětlený rozptyl: {pca.explained_variance_ratio_.sum():.4f}")

# 2. Detekce anomálií (Isolation Forest)
print("Trénuji Isolation Forest na redukovaných datech...")
clf = IForest(contamination=0.01, n_jobs=-1, random_state=42)
clf.fit(X_pca)
print("Model Isolation Forest natrénován.")

# 3. Mapování výsledků na ID protokolů
print("Propojuji vypočtená skóre s identifikátory...")
vysledky_map = id.with_columns([
    pl.Series("anomaly_score", clf.decision_scores_),
    pl.Series("is_anomaly", clf.labels_)
])

# 4. Filtrace a spojení s původním dataframe
print("Filtruji anomálie a spojuji s původními daty...")
anomali_ids = vysledky_map.filter(pl.col("is_anomaly") == 1).select("CisloProtokolu")

# Vytvoření finálního dataframe se všemi původními sloupci
df_anomalie = df.join(
    anomali_ids, on="CisloProtokolu", how="inner"
).join(
    vysledky_map.select(["CisloProtokolu", "anomaly_score"]), 
    on="CisloProtokolu", 
    how="left"
).sort("anomaly_score", descending=True)

print(f"Hotovo. Celkem nalezeno {df_anomalie.height} anomálií.")
short_display(df_anomalie)

Zahajuji PCA pro redukci dimenzionality...
PCA dokončeno. Vysvětlený rozptyl: 0.2823
Trénuji Isolation Forest na redukovaných datech...
Model Isolation Forest natrénován.
Propojuji vypočtená skóre s identifikátory...
Filtruji anomálie a spojuji s původními daty...
Hotovo. Celkem nalezeno 1000 anomálií.
(1000, 224)


,CisloProtokolu,MericiPristroj_Vyrobce,MericiPristroj_Typ,MericiPristroj_Verze,MericiPristroj_OBD,MericiPristroj_VerzeSoftware,Vysledek_VisualniKontrola,Vysledek_Readiness,Vysledek_RidiciJednotkaStav,Vysledek_Mil,...,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Zahajeni_Sin,Zahajeni_Cos,anomaly_score
0,CZ-470118-19-04-0286,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,1.0,2.0,...,False,False,118426,False,12.966667,6.450000,19.085558,-0.811165,-0.584817,0.057636
1,CZ-470118-20-10-0106,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,1.0,2.0,...,False,False,141970,False,8.716667,7.683333,21.445585,-0.527029,-0.849848,0.057636
2,CZ-470118-20-07-0149,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,1.0,2.0,...,False,False,167689,False,9.816667,7.800000,20.164271,0.986757,-0.162207,0.054915
3,CZ-470118-19-04-0023,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,1.0,2.0,...,False,False,50513,False,22.350000,7.983333,19.101985,-0.659225,-0.751945,0.054488
4,CZ-470118-21-09-0294,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,1.0,2.0,...,False,False,78268,False,10.833333,8.983333,26.724162,-0.393223,-0.919443,0.051810
5,CZ-470118-19-07-0063,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,KTS540,V1.21,True,True,1.0,2.0,...,False,False,71277,False,9.000000,7.050000,6.017796,0.239441,-0.970911,0.050467
6,CZ-470118-19-04-0184,Bosch,BEA450_460,BEA V2.02-Mobil 38710554 / AMM 000-B6 F54B,None,V1.21,True,None,NaN,2.0,...,False,False,62251,False,101.550000,10.183333,37.897331,0.729703,-0.683764,0.049495
7,CZ-470403-19-08-0232,BOSCH,BEA450,SW-BEA-PC CZ V1.21,ne,CZ V1.21,True,None,NaN,NaN,...,False,False,168076,False,21.483333,20.733333,21.560575,0.818650,-0.574293,0.049262
8,CZ-420820-23-06-0355,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,None,V2.5.2 09/2022,True,None,1.0,2.0,...,False,False,127886,False,94.033333,9.050000,22.012320,0.496839,-0.867843,0.048905
9,CZ-420820-23-06-0195,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,None,V2.5.2 09/2022,True,None,1.0,2.0,...,False,False,134947,False,117.633333,14.783333,22.113621,-0.293302,-0.956020,0.048781


In [20]:
short_display(df.filter(pl.col('Benzin_PocetVyusteni') == 1))
rucni_cols = [col for col in df_anomalie.columns if 'RucniZadani' in col]
short_display(df_anomalie.filter(~pl.any_horizontal(pl.col(rucni_cols).fill_null(False) == True)).filter(pl.col('EmisniSystem') == 'Rizeny_Obd'))

(54122, 223)


,CisloProtokolu,MericiPristroj_Vyrobce,MericiPristroj_Typ,MericiPristroj_Verze,MericiPristroj_OBD,MericiPristroj_VerzeSoftware,Vysledek_VisualniKontrola,Vysledek_Readiness,Vysledek_RidiciJednotkaStav,Vysledek_Mil,...,TechnickaCast_Pritomno,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Zahajeni_Sin,Zahajeni_Cos
0,CZ-410408-19-12-0020,BOSCH,BEA050,SW-BEA-PC CZ V1.21,ano,CZ V1.21,True,True,1.0,2.0,...,False,False,False,126198,False,12.533333,8.533333,5.957563,0.282365,-0.959307
1,CZ-450402-20-08-0031,BRAIN BEE,AGS-200,-,ano,2019.0.1,True,False,1.0,2.0,...,False,False,False,164727,False,12.766667,8.200000,11.107461,-0.371479,-0.928442
2,CZ-480764-21-06-0256,BOSCH,BEA050,SW-BEA-PC CZ V1.21,ano,CZ V1.21,True,True,1.0,2.0,...,False,False,False,95131,False,14.483333,14.083333,4.002738,-0.558042,-0.829812
3,CZ-420940-21-12-0758,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL VCI 1000,V1.41 10/2014,True,True,1.0,2.0,...,False,False,False,81654,False,173.433333,10.350000,3.983573,0.390166,-0.920744
4,CZ-410533-24-09-0491,Bosch,BEA060,BEA060 V1.14 6E795EEC / AMM B6 F54B /,KTS560,1.1,True,True,1.0,2.0,...,False,False,False,88164,False,7.183333,6.216667,3.994524,0.292859,-0.956156
5,CZ-110903-25-07-0123,ATAL s.r.o.,AT505,-,R+OBD,3.08.3,True,True,1.0,2.0,...,False,False,False,50849,False,9.066667,7.750000,8.260096,0.983845,-0.179021
6,CZ-3765-25-08-0345,BRAIN BEE,-,-,-,-,True,True,1.0,1.0,...,True,False,False,93353,False,56.183333,17.883333,65.798768,0.635869,-0.771797
7,CZ-471117-19-05-0010,BRAIN BEE,AGS-200,-,None,2019.0.1,True,None,1.0,2.0,...,False,False,False,141952,False,116.883333,13.216667,18.190281,0.938796,-0.344474
8,CZ-470530-22-01-0015,ActiaCZ,AT505,sw: 2.04.3,ne,sw: 2.04.3,True,None,NaN,2.0,...,False,False,False,135492,False,13.100000,7.283333,22.157426,0.528785,-0.848756
9,CZ-460214-21-08-1048,Bosch,BEA050,BEA V2.02 3AA30592 / AMM 000-B6 F54B,KTS540,V1.21,True,True,1.0,2.0,...,False,False,False,53773,False,15.483333,14.833333,3.312799,-0.970627,-0.240589


(357, 224)


,CisloProtokolu,MericiPristroj_Vyrobce,MericiPristroj_Typ,MericiPristroj_Verze,MericiPristroj_OBD,MericiPristroj_VerzeSoftware,Vysledek_VisualniKontrola,Vysledek_Readiness,Vysledek_RidiciJednotkaStav,Vysledek_Mil,...,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Zahajeni_Sin,Zahajeni_Cos,anomaly_score
0,CZ-470118-24-08-0110,Bosch,BEA 950,/ BEA070 V1.19 CFFE0424,KTS540,1.0,True,True,1,2,...,False,False,283417,False,12.550000,10.450000,16.114990,0.921602,-0.388136,0.039640
1,CZ-420820-25-10-0137,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V3.0.5 01/2025,True,True,1,2,...,False,False,162416,False,47.716667,7.816667,10.288843,-0.732386,-0.680890,0.039013
2,CZ-420820-25-10-0128,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V3.0.5 01/2025,True,True,1,2,...,False,False,74725,False,48.316667,13.066667,10.390144,-0.362091,-0.932143,0.039013
3,CZ-420820-24-12-0045,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,94961,False,55.816667,7.016667,6.015058,0.869295,-0.494293,0.038180
4,CZ-420820-23-06-0203,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,143695,False,57.933333,12.216667,11.039014,0.964961,-0.262392,0.037338
5,CZ-420820-24-09-0112,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,False,1,2,...,False,False,154253,False,15.950000,13.416667,22.280630,-0.169130,-0.985594,0.036465
6,CZ-420820-24-04-0170,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,157839,False,33.616667,12.050000,24.071184,0.627806,-0.778370,0.036421
7,CZ-420820-23-09-0002,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,123988,False,15.266667,8.183333,9.932923,0.960061,-0.279790,0.036388
8,CZ-420820-24-10-0043,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,138975,False,11.816667,8.633333,19.260780,0.339567,-0.940582,0.036050
9,CZ-420820-24-03-0192,AVL DiTEST GmbH,AVL Gas 1000,V1.41 10/2014,AVL OBD 1000,V2.5.2 09/2022,True,True,1,2,...,False,False,71564,False,22.066667,16.100000,9.242984,0.915265,-0.402852,0.034682


# Anomalie
- rucni zadani
- bez OBD

In [18]:
short_display(df.filter(pl.col('Benzin_PocetVyusteni') == 1).filter(~pl.any_horizontal(pl.col(rucni_cols) == True)))

(0, 223)


,CisloProtokolu,MericiPristroj_Vyrobce,MericiPristroj_Typ,MericiPristroj_Verze,MericiPristroj_OBD,MericiPristroj_VerzeSoftware,Vysledek_VisualniKontrola,Vysledek_Readiness,Vysledek_RidiciJednotkaStav,Vysledek_Mil,...,TechnickaCast_Pritomno,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Zahajeni_Sin,Zahajeni_Cos
